In [1]:
# -*- coding: utf-8 -*-

05_evaluation.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1u0POb2DZqXrRPEScjj1GjB8uqoIVUC48

# Evaluation and Standardized Comparison Table

In this notebook, we compare the four recommenders developed in the project:

- Non-Personalized
- Collaborative Filtering
- Content-Based
- Context-Aware

The final output is a standardized comparison table with the following columns:

Approach | RMSE | MAE | Precision@K | Recall@K | NDCG | Coverage | Diversity | Serendipity | Context

Contributors: Andrés Ramírez, Santiago Llorca

In [2]:
import pandas as pd
import numpy as np
from itertools import combinations
from sklearn.metrics.pairwise import cosine_similarity

## 1. Load Exported Predictions

In [3]:
np_df = pd.read_csv("predictions_non_personalized.csv")
cf_df = pd.read_csv("predictions_cf.csv")
content_df = pd.read_csv("predictions_content.csv")
context_df = pd.read_csv("predictions_context.csv")

print("Non-Personalized:", np_df.shape)
print("Collaborative Filtering:", cf_df.shape)
print("Content-Based:", content_df.shape)
print("Context-Aware:", context_df.shape)

Non-Personalized: (20168, 5)
Collaborative Filtering: (20168, 5)
Content-Based: (20168, 5)
Context-Aware: (20168, 7)


## 2. Merge All Prediction Files

In [4]:
eval_df = np_df.merge(cf_df, on=["userId", "movieId", "rating"], how="inner")
eval_df = eval_df.merge(content_df, on=["userId", "movieId", "rating"], how="inner")
eval_df = eval_df.merge(context_df, on=["userId", "movieId", "rating"], how="inner")

print("Merged evaluation dataframe:", eval_df.shape)
eval_df.head()

Merged evaluation dataframe: (20168, 13)


,userId,movieId,rating,predicted_rating_non_personalized,score_non_personalized,score_cf,predicted_rating_cf,score_content,predicted_rating_content,context,context_rich,score_context,predicted_rating_context
0,495,72998,5.0,3.662521,3.662521,4.474270,4.474270,0.260112,4.783460,weekday,weekday_morning,0.260112,4.783460
1,495,2762,4.5,3.866102,3.866102,4.915041,4.915041,0.340262,4.660192,weekday,weekday_morning,0.340262,4.660192
2,495,4993,0.5,4.093642,4.093642,4.763956,4.763956,0.108660,4.825495,weekday,weekday_morning,0.108660,4.825495
3,495,89492,5.0,3.632456,3.632456,4.792751,4.792751,0.487505,4.680301,weekday,weekday_morning,0.487505,4.680301
4,495,2028,4.5,4.130230,4.130230,4.866406,4.866406,0.434230,4.641142,weekday,weekday_morning,0.434230,4.641142


## 3. Load Original Data for Supporting Metrics

In [5]:
try:
    ratings = pd.read_csv("ml-latest-small/ratings.csv")
    movies = pd.read_csv("ml-latest-small/movies.csv")
except FileNotFoundError:
    ratings = pd.read_csv("ratings.csv")
    movies = pd.read_csv("movies.csv")

ratings = ratings.sort_values("timestamp").copy()
cutoff = int(len(ratings) * 0.8)

train = ratings.iloc[:cutoff].copy()
test = ratings.iloc[cutoff:].copy()

print("Train size:", train.shape)
print("Test size:", test.shape)

Train size: (80668, 4)
Test size: (20168, 4)


## 4. Build Item Similarity Matrix Again

This is needed for diversity and serendipity.

In [6]:
movies_features = movies.copy()
movies_features["genres"] = movies_features["genres"].str.replace("|", " ", regex=False)

genre_matrix = movies_features["genres"].str.get_dummies(sep=" ")
genre_matrix.index = movies_features["movieId"]

item_similarity = pd.DataFrame(
    cosine_similarity(genre_matrix),
    index=genre_matrix.index,
    columns=genre_matrix.index
)

item_similarity.head()

movieId,1,2,3,4,5,6,7,8,9,10,...,193565,193567,193571,193573,193579,193581,193583,193585,193587,193609
movieId,,,,,,,,,,,,,,,,,,,,,
1,1.000000,0.774597,0.316228,0.258199,0.447214,0.0,0.316228,0.632456,0.0,0.258199,...,0.447214,0.316228,0.316228,0.447214,0.0,0.670820,0.774597,0.00000,0.316228,0.447214
2,0.774597,1.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.816497,0.0,0.333333,...,0.000000,0.000000,0.000000,0.000000,0.0,0.288675,0.333333,0.00000,0.000000,0.000000
3,0.316228,0.000000,1.000000,0.816497,0.707107,0.0,1.000000,0.000000,0.0,0.000000,...,0.353553,0.000000,0.500000,0.000000,0.0,0.353553,0.408248,0.00000,0.000000,0.707107
4,0.258199,0.000000,0.816497,1.000000,0.577350,0.0,0.816497,0.000000,0.0,0.000000,...,0.288675,0.408248,0.816497,0.000000,0.0,0.288675,0.333333,0.57735,0.000000,0.577350
5,0.447214,0.000000,0.707107,0.577350,1.000000,0.0,0.707107,0.000000,0.0,0.000000,...,0.500000,0.000000,0.707107,0.000000,0.0,0.500000,0.577350,0.00000,0.000000,1.000000


## 5. Global Parameters

In [7]:
TOP_K = 10
RELEVANCE_THRESHOLD = 4.0
catalog_size = movies["movieId"].nunique()

## 6. Metric Functions

In [8]:
def rmse_metric(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    return np.sqrt(np.mean((y_true - y_pred) ** 2))


def mae_metric(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    return np.mean(np.abs(y_true - y_pred))


def precision_at_k(data, score_column, k=10, relevance_threshold=4.0):
    precisions = []

    for user in data["userId"].unique():
        user_data = data[data["userId"] == user].sort_values(score_column, ascending=False)
        top_k = user_data.head(k)

        if len(top_k) == 0:
            continue

        relevant = (top_k["rating"] >= relevance_threshold).sum()
        precisions.append(relevant / k)

    return np.mean(precisions) if precisions else np.nan


def recall_at_k(data, score_column, k=10, relevance_threshold=4.0):
    recalls = []

    for user in data["userId"].unique():
        user_data = data[data["userId"] == user]
        total_relevant = (user_data["rating"] >= relevance_threshold).sum()

        if total_relevant == 0:
            continue

        top_k = user_data.sort_values(score_column, ascending=False).head(k)
        relevant_in_top_k = (top_k["rating"] >= relevance_threshold).sum()

        recalls.append(relevant_in_top_k / total_relevant)

    return np.mean(recalls) if recalls else np.nan


def ndcg_at_k(data, score_column, k=10, relevance_threshold=4.0):
    ndcgs = []

    for user in data["userId"].unique():
        user_data = data[data["userId"] == user].sort_values(score_column, ascending=False).head(k)
        gains = (user_data["rating"] >= relevance_threshold).astype(int).values

        dcg = 0
        for i, rel in enumerate(gains, start=1):
            dcg += rel / np.log2(i + 1)

        ideal_gains = sorted(gains, reverse=True)
        idcg = 0
        for i, rel in enumerate(ideal_gains, start=1):
            idcg += rel / np.log2(i + 1)

        if idcg == 0:
            continue

        ndcgs.append(dcg / idcg)

    return np.mean(ndcgs) if ndcgs else np.nan


def coverage_at_k(data, score_column, catalog_size, k=10):
    recommended_items = set()

    for user in data["userId"].unique():
        user_data = data[data["userId"] == user].sort_values(score_column, ascending=False).head(k)
        recommended_items.update(user_data["movieId"].tolist())

    return len(recommended_items) / catalog_size if catalog_size > 0 else np.nan


def diversity_at_k(data, score_column, item_similarity, k=10):
    diversities = []

    for user in data["userId"].unique():
        top_k_items = (
            data[data["userId"] == user]
            .sort_values(score_column, ascending=False)
            .head(k)["movieId"]
            .tolist()
        )

        if len(top_k_items) < 2:
            continue

        pair_dissimilarities = []

        for i, j in combinations(top_k_items, 2):
            if i in item_similarity.index and j in item_similarity.columns:
                sim = item_similarity.loc[i, j]
                pair_dissimilarities.append(1 - sim)

        if pair_dissimilarities:
            diversities.append(np.mean(pair_dissimilarities))

    return np.mean(diversities) if diversities else np.nan


def serendipity_at_k(data, score_column, train, item_similarity, k=10, relevance_threshold=4.0):
    serendipities = []

    for user in data["userId"].unique():
        user_train_items = train[train["userId"] == user]["movieId"].unique().tolist()

        top_k = (
            data[data["userId"] == user]
            .sort_values(score_column, ascending=False)
            .head(k)
        )

        top_k = top_k[top_k["rating"] >= relevance_threshold]

        if top_k.empty:
            continue

        user_serendipity = []

        for rec_item in top_k["movieId"]:
            if rec_item not in item_similarity.index or len(user_train_items) == 0:
                continue

            seen_sims = []
            for seen_item in user_train_items:
                if seen_item in item_similarity.columns:
                    seen_sims.append(item_similarity.loc[rec_item, seen_item])

            if seen_sims:
                novelty = 1 - max(seen_sims)
                user_serendipity.append(novelty)

        if user_serendipity:
            serendipities.append(np.mean(user_serendipity))

    return np.mean(serendipities) if serendipities else np.nan

## 7. Compute Metrics for Each Approach

In [9]:
rmse_np = rmse_metric(eval_df["rating"], eval_df["predicted_rating_non_personalized"])
mae_np = mae_metric(eval_df["rating"], eval_df["predicted_rating_non_personalized"])

rmse_cf = rmse_metric(eval_df["rating"], eval_df["predicted_rating_cf"])
mae_cf = mae_metric(eval_df["rating"], eval_df["predicted_rating_cf"])

rmse_content = rmse_metric(eval_df["rating"], eval_df["predicted_rating_content"])
mae_content = mae_metric(eval_df["rating"], eval_df["predicted_rating_content"])

rmse_context = rmse_metric(eval_df["rating"], eval_df["predicted_rating_context"])
mae_context = mae_metric(eval_df["rating"], eval_df["predicted_rating_context"])

precision_np = precision_at_k(eval_df, "score_non_personalized", k=TOP_K, relevance_threshold=RELEVANCE_THRESHOLD)
recall_np = recall_at_k(eval_df, "score_non_personalized", k=TOP_K, relevance_threshold=RELEVANCE_THRESHOLD)
ndcg_np = ndcg_at_k(eval_df, "score_non_personalized", k=TOP_K, relevance_threshold=RELEVANCE_THRESHOLD)
coverage_np = coverage_at_k(eval_df, "score_non_personalized", catalog_size, k=TOP_K)
diversity_np = diversity_at_k(eval_df, "score_non_personalized", item_similarity, k=TOP_K)
serendipity_np = serendipity_at_k(eval_df, "score_non_personalized", train, item_similarity, k=TOP_K, relevance_threshold=RELEVANCE_THRESHOLD)

precision_cf = precision_at_k(eval_df, "score_cf", k=TOP_K, relevance_threshold=RELEVANCE_THRESHOLD)
recall_cf = recall_at_k(eval_df, "score_cf", k=TOP_K, relevance_threshold=RELEVANCE_THRESHOLD)
ndcg_cf = ndcg_at_k(eval_df, "score_cf", k=TOP_K, relevance_threshold=RELEVANCE_THRESHOLD)
coverage_cf = coverage_at_k(eval_df, "score_cf", catalog_size, k=TOP_K)
diversity_cf = diversity_at_k(eval_df, "score_cf", item_similarity, k=TOP_K)
serendipity_cf = serendipity_at_k(eval_df, "score_cf", train, item_similarity, k=TOP_K, relevance_threshold=RELEVANCE_THRESHOLD)

precision_content = precision_at_k(eval_df, "score_content", k=TOP_K, relevance_threshold=RELEVANCE_THRESHOLD)
recall_content = recall_at_k(eval_df, "score_content", k=TOP_K, relevance_threshold=RELEVANCE_THRESHOLD)
ndcg_content = ndcg_at_k(eval_df, "score_content", k=TOP_K, relevance_threshold=RELEVANCE_THRESHOLD)
coverage_content = coverage_at_k(eval_df, "score_content", catalog_size, k=TOP_K)
diversity_content = diversity_at_k(eval_df, "score_content", item_similarity, k=TOP_K)
serendipity_content = serendipity_at_k(eval_df, "score_content", train, item_similarity, k=TOP_K, relevance_threshold=RELEVANCE_THRESHOLD)

precision_context = precision_at_k(eval_df, "score_context", k=TOP_K, relevance_threshold=RELEVANCE_THRESHOLD)
recall_context = recall_at_k(eval_df, "score_context", k=TOP_K, relevance_threshold=RELEVANCE_THRESHOLD)
ndcg_context = ndcg_at_k(eval_df, "score_context", k=TOP_K, relevance_threshold=RELEVANCE_THRESHOLD)
coverage_context = coverage_at_k(eval_df, "score_context", catalog_size, k=TOP_K)
diversity_context = diversity_at_k(eval_df, "score_context", item_similarity, k=TOP_K)
serendipity_context = serendipity_at_k(eval_df, "score_context", train, item_similarity, k=TOP_K, relevance_threshold=RELEVANCE_THRESHOLD)

## 8. Final Standardized Comparison Table

In [10]:
results = pd.DataFrame({
    "Approach": ["Non-Personalized", "Collaborative Filtering", "Content-Based", "Context-Aware"],
    "RMSE": [rmse_np, rmse_cf, rmse_content, rmse_context],
    "MAE": [mae_np, mae_cf, mae_content, mae_context],
    "Precision@K": [precision_np, precision_cf, precision_content, precision_context],
    "Recall@K": [recall_np, recall_cf, recall_content, recall_context],
    "NDCG": [ndcg_np, ndcg_cf, ndcg_content, ndcg_context],
    "Coverage": [coverage_np, coverage_cf, coverage_content, coverage_context],
    "Diversity": [diversity_np, diversity_cf, diversity_content, diversity_context],
    "Serendipity": [serendipity_np, serendipity_cf, serendipity_content, serendipity_context],
    "Context": ["No", "No", "No", "Weekday/Weekend"]
})

results.round(4)

,Approach,RMSE,MAE,Precision@K,Recall@K,NDCG,Coverage,Diversity,Serendipity,Context
0,Non-Personalized,1.0268,0.8010,0.7000,0.2420,0.8894,0.0253,0.6598,0.0992,No
1,Collaborative Filtering,1.0714,0.8444,0.6069,0.2260,0.8856,0.0696,0.6629,0.1083,No
2,Content-Based,1.1072,0.8661,0.5957,0.2217,0.8654,0.0727,0.6044,0.0660,No
3,Context-Aware,1.1066,0.8659,0.5957,0.2244,0.8675,0.0734,0.6149,0.0673,Weekday/Weekend


## 9 A/B Testing

To simulate an online comparison, we would test the two strongest candidate models:

- Variant A: Non-Personalized Recommender
- Variant B: Collaborative Filtering Recommender

Users would be randomly assigned to one of the two variants.

The main KPI would be Hit Rate@10, defined as whether at least one relevant item (rating >= 4) appears in the user’s top-10 recommendations.

Secondary KPIs could include Precision@10 and NDCG@10.

The winning model would be the one that achieves the best balance between user relevance and recommendation quality.

In [11]:
np.random.seed(1)

ab_users = pd.DataFrame({
    "userId": eval_df["userId"].unique(),
    "group": np.random.choice(["A", "B"], size=eval_df["userId"].nunique())
})

ab_df = eval_df.merge(ab_users, on="userId", how="left")
ab_df.head()

def hit_rate_at_k(data, score_column, k=10, relevance_threshold=4.0):
    hits = []

    for user in data["userId"].unique():
        top_k = (
            data[data["userId"] == user]
            .sort_values(score_column, ascending=False)
            .head(k)
        )
        hits.append(int((top_k["rating"] >= relevance_threshold).any()))

    return np.mean(hits)

group_a = ab_df[ab_df["group"] == "A"]
group_b = ab_df[ab_df["group"] == "B"]

hit_rate_a = hit_rate_at_k(group_a, "score_non_personalized", k=10, relevance_threshold=4.0)
hit_rate_b = hit_rate_at_k(group_b, "score_cf", k=10, relevance_threshold=4.0)

ab_results = pd.DataFrame({
    "Variant": ["A", "B"],
    "Model": ["Non-Personalized", "Collaborative Filtering"],
    "HitRate@10": [hit_rate_a, hit_rate_b]
})

ab_results

,Variant,Model,HitRate@10
0,A,Non-Personalized,0.980392
1,B,Collaborative Filtering,0.969231


## 10. Hybrid Recommender

We combine the four recommenders into a weighted hybrid.
Each score is first min-max normalised to [0, 1] so that different scales
do not dominate the final score.  The weights are configurable and sum to 1.

Formula:
    score_hybrid = w_np  * norm(score_non_personalized)
                 + w_cf  * norm(score_cf)
                 + w_cb  * norm(score_content)
                 + w_ctx * norm(score_context)

In [12]:
def minmax_normalize(series):
    """Normalise a pandas Series to [0, 1]. Returns 0.5 for constant series."""
    mn, mx = series.min(), series.max()
    if mx == mn:
        return pd.Series(0.5, index=series.index)
    return (series - mn) / (mx - mn)

# --- Configurable weights (must sum to 1) ---
# Default: CF and content carry most weight; non-personalized is a small floor.
HYBRID_WEIGHTS = {
    "w_np":  0.10,   # non-personalized
    "w_cf":  0.40,   # collaborative filtering
    "w_cb":  0.30,   # content-based
    "w_ctx": 0.20,   # context-aware
}

assert abs(sum(HYBRID_WEIGHTS.values()) - 1.0) < 1e-6, "Weights must sum to 1"

# Normalise each score column on the evaluation set
eval_df["norm_np"]      = minmax_normalize(eval_df["score_non_personalized"])
eval_df["norm_cf"]      = minmax_normalize(eval_df["score_cf"])
eval_df["norm_content"] = minmax_normalize(eval_df["score_content"])
eval_df["norm_context"] = minmax_normalize(eval_df["score_context"])

# Compute hybrid score
eval_df["score_hybrid"] = (
    HYBRID_WEIGHTS["w_np"]  * eval_df["norm_np"]
  + HYBRID_WEIGHTS["w_cf"]  * eval_df["norm_cf"]
  + HYBRID_WEIGHTS["w_cb"]  * eval_df["norm_content"]
  + HYBRID_WEIGHTS["w_ctx"] * eval_df["norm_context"]
)

# Predicted rating hybrid: weighted average of the four predicted ratings
eval_df["predicted_rating_hybrid"] = (
    HYBRID_WEIGHTS["w_np"]  * eval_df["predicted_rating_non_personalized"]
  + HYBRID_WEIGHTS["w_cf"]  * eval_df["predicted_rating_cf"]
  + HYBRID_WEIGHTS["w_cb"]  * eval_df["predicted_rating_content"]
  + HYBRID_WEIGHTS["w_ctx"] * eval_df["predicted_rating_context"]
).clip(0.5, 5.0)

print("Hybrid scores computed.")
eval_df[["userId", "movieId", "rating", "score_hybrid", "predicted_rating_hybrid"]].head()

Hybrid scores computed.


,userId,movieId,rating,score_hybrid,predicted_rating_hybrid
0,495,72998,5.0,0.610390,4.547690
1,495,2762,4.5,0.718100,4.682722
2,495,4993,0.5,0.544934,4.727694
3,495,89492,5.0,0.804607,4.620497
4,495,2028,4.5,0.795317,4.680157


## 11. Cold-Start Strategy

Users with very little history cannot be served well by CF (not enough overlap
with other users) or context-aware (not enough contextual interactions).

We define a tiered strategy based on the number of ratings a user has in train:

  - Cold  (< 5 ratings)  → rely on non-personalized + content-based
  - Warm  (5–20 ratings) → balanced mix
  - Hot   (> 20 ratings) → CF + context carry more weight

The cold-start score is computed on the fly using the same normalised columns
already present in eval_df.

In [13]:
# Count interactions per user in train
user_activity = train.groupby("userId")["rating"].count().rename("n_train_ratings")
eval_df = eval_df.merge(user_activity, on="userId", how="left")
eval_df["n_train_ratings"] = eval_df["n_train_ratings"].fillna(0).astype(int)

# Configurable thresholds
COLD_THRESHOLD = 5    # fewer than this → cold user
WARM_THRESHOLD = 20   # between cold and this → warm user

def cold_start_weights(n_ratings):
    """Return (w_np, w_cf, w_cb, w_ctx) based on user activity level."""
    if n_ratings < COLD_THRESHOLD:
        # Cold: no reliable CF signal — lean on content + non-personalized
        return (0.30, 0.10, 0.50, 0.10)
    elif n_ratings < WARM_THRESHOLD:
        # Warm: balanced
        return (0.15, 0.30, 0.35, 0.20)
    else:
        # Hot: CF and context are reliable
        return (0.10, 0.40, 0.30, 0.20)

def compute_cs_score(row):
    w_np, w_cf, w_cb, w_ctx = cold_start_weights(row["n_train_ratings"])
    return (
        w_np  * row["norm_np"]
      + w_cf  * row["norm_cf"]
      + w_cb  * row["norm_content"]
      + w_ctx * row["norm_context"]
    )

eval_df["score_hybrid_cs"] = eval_df.apply(compute_cs_score, axis=1)

# Summary of user tiers
tier_counts = pd.cut(
    eval_df.drop_duplicates("userId")["n_train_ratings"],
    bins=[-1, COLD_THRESHOLD - 1, WARM_THRESHOLD - 1, float("inf")],
    labels=["Cold", "Warm", "Hot"]
).value_counts()

print("User tiers in evaluation set:")
print(tier_counts)

User tiers in evaluation set:
n_train_ratings
Cold    88
Hot     27
Warm     1
Name: count, dtype: int64


In [14]:
# Recreate context features (same as Notebook 04)

train["datetime"] = pd.to_datetime(train["timestamp"], unit="s")
train["hour"] = train["datetime"].dt.hour
train["dayofweek"] = train["datetime"].dt.dayofweek

train["context"] = np.where(train["dayofweek"] >= 5, "weekend", "weekday")

def get_time_of_day(hour):
    if 6 <= hour < 12:
        return "morning"
    elif 12 <= hour < 18:
        return "afternoon"
    elif 18 <= hour < 24:
        return "evening"
    else:
        return "night"

train["time_of_day"] = train["hour"].apply(get_time_of_day)
train["context_rich"] = train["context"] + "_" + train["time_of_day"]

## 12. Explainability Layer

For a given user and movie, we generate a human-readable explanation of why
the hybrid recommender suggested it.  The function reuses existing data
structures already computed in this notebook.

Three explanation types:
  1. Content signal   — similar genres to movies the user liked
  2. CF signal        — similar users also rated it highly
  3. Context signal   — the user rated similar movies in this time context

In [15]:
def explain_recommendation(user_id, movie_id, eval_df, train,
                            item_similarity, user_item_matrix,
                            user_similarity_cosine, movies,
                            context_col="context_rich", top_n_reasons=3):
    """
    Returns a list of plain-text explanation strings for a recommendation.

    Parameters
    ----------
    user_id            : int
    movie_id           : int
    eval_df            : merged evaluation dataframe (must contain score columns)
    train              : training ratings dataframe
    item_similarity    : genre-based item-item cosine similarity DataFrame
    user_item_matrix   : user-item matrix from CF notebook (pass if available)
    user_similarity_cosine : user-user cosine similarity DataFrame (from CF)
    movies             : movies metadata DataFrame
    context_col        : context column name ('context' or 'context_rich')
    top_n_reasons      : max number of reasons to return
    """
    reasons = []

    movie_title = movies[movies["movieId"] == movie_id]["title"].values
    movie_title = movie_title[0] if len(movie_title) > 0 else str(movie_id)

    # --- 1. Content-based reason ---
    user_train = train[train["userId"] == user_id]
    liked = user_train[user_train["rating"] >= 4.0]["movieId"].tolist()

    if movie_id in item_similarity.index and liked:
        sims = {m: item_similarity.loc[movie_id, m]
                for m in liked if m in item_similarity.columns}
        if sims:
            best_match_id = max(sims, key=sims.get)
            best_sim = sims[best_match_id]
            best_title = movies[movies["movieId"] == best_match_id]["title"].values
            best_title = best_title[0] if len(best_title) > 0 else str(best_match_id)
            if best_sim > 0.3:
                reasons.append(
                    f"[Content] Similar genres to '{best_title}', "
                    f"which you rated highly (genre similarity: {best_sim:.2f})."
                )

    # --- 2. CF reason ---
    if (user_item_matrix is not None
            and user_id in user_similarity_cosine.index
            and movie_id in user_item_matrix.columns):
        sim_scores = user_similarity_cosine.loc[user_id].sort_values(ascending=False)
        top_neighbors = sim_scores[sim_scores > 0].head(10).index
        neighbor_ratings = user_item_matrix.loc[
            user_item_matrix.index.isin(top_neighbors), movie_id
        ].dropna()
        if not neighbor_ratings.empty:
            avg_neighbor_rating = neighbor_ratings.mean()
            n_neighbors = len(neighbor_ratings)
            reasons.append(
                f"[CF] {n_neighbors} similar user(s) rated this movie "
                f"with an average of {avg_neighbor_rating:.1f}/5."
            )

    # --- 3. Context reason ---
    row = eval_df[(eval_df["userId"] == user_id) & (eval_df["movieId"] == movie_id)]
    if not row.empty and context_col in row.columns:
        ctx = row[context_col].values[0]
        ctx_score = row["score_context"].values[0]
        n_ctx_ratings = train[
            (train["userId"] == user_id) & (train[context_col] == ctx)
        ].shape[0]
        if ctx_score > 0 and n_ctx_ratings > 0:
            reasons.append(
                f"[Context] You have {n_ctx_ratings} ratings in the "
                f"'{ctx}' context, and this movie fits your pattern for that time."
            )

    if not reasons:
        reasons.append(
            "[Fallback] Recommended based on overall popularity "
            "and your general rating history."
        )

    return reasons[:top_n_reasons]


# --- Example usage ---
# Pick a user and movie that appear in the evaluation set
sample_row = eval_df.iloc[0]
sample_user  = int(sample_row["userId"])
sample_movie = int(sample_row["movieId"])

print(f"Explaining recommendation of movie {sample_movie} to user {sample_user}:\n")

# Note: user_item_matrix and user_similarity_cosine are defined in notebook 02.
# In this standalone notebook we pass None to skip the CF explanation gracefully.
explanations = explain_recommendation(
    user_id=sample_user,
    movie_id=sample_movie,
    eval_df=eval_df,
    train=train,
    item_similarity=item_similarity,
    user_item_matrix=None,           # set to user_item_matrix if imported from nb02
    user_similarity_cosine=None,     # set to user_similarity_cosine if imported
    movies=movies,
    context_col="context_rich"
)

for e in explanations:
    print(" •", e)

Explaining recommendation of movie 72998 to user 495:

 • [Content] Similar genres to 'Star Wars: Episode VII - The Force Awakens (2015)', which you rated highly (genre similarity: 0.89).
 • [Context] You have 29 ratings in the 'weekday_morning' context, and this movie fits your pattern for that time.


## 13. Updated Final Standardized Comparison Table

We now include the Hybrid recommender alongside the original four approaches.

In [16]:
rmse_hybrid = rmse_metric(eval_df["rating"], eval_df["predicted_rating_hybrid"])
mae_hybrid  = mae_metric(eval_df["rating"],  eval_df["predicted_rating_hybrid"])

precision_hybrid  = precision_at_k(eval_df,  "score_hybrid", k=TOP_K, relevance_threshold=RELEVANCE_THRESHOLD)
recall_hybrid     = recall_at_k(eval_df,     "score_hybrid", k=TOP_K, relevance_threshold=RELEVANCE_THRESHOLD)
ndcg_hybrid       = ndcg_at_k(eval_df,       "score_hybrid", k=TOP_K, relevance_threshold=RELEVANCE_THRESHOLD)
coverage_hybrid   = coverage_at_k(eval_df,   "score_hybrid", catalog_size, k=TOP_K)
diversity_hybrid  = diversity_at_k(eval_df,  "score_hybrid", item_similarity, k=TOP_K)
serendipity_hybrid = serendipity_at_k(eval_df, "score_hybrid", train, item_similarity,
                                       k=TOP_K, relevance_threshold=RELEVANCE_THRESHOLD)

results_full = pd.DataFrame({
    "Approach": [
        "Non-Personalized", "Collaborative Filtering",
        "Content-Based", "Context-Aware", "Hybrid"
    ],
    "RMSE": [rmse_np, rmse_cf, rmse_content, rmse_context, rmse_hybrid],
    "MAE":  [mae_np,  mae_cf,  mae_content,  mae_context,  mae_hybrid],
    "Precision@K": [precision_np, precision_cf, precision_content, precision_context, precision_hybrid],
    "Recall@K":    [recall_np,    recall_cf,    recall_content,    recall_context,    recall_hybrid],
    "NDCG":        [ndcg_np,      ndcg_cf,      ndcg_content,      ndcg_context,      ndcg_hybrid],
    "Coverage":    [coverage_np,  coverage_cf,  coverage_content,  coverage_context,  coverage_hybrid],
    "Diversity":   [diversity_np, diversity_cf, diversity_content, diversity_context, diversity_hybrid],
    "Serendipity": [serendipity_np, serendipity_cf, serendipity_content,
                    serendipity_context, serendipity_hybrid],
    "Context":     ["No", "No", "No", "Weekday/Weekend + Time", "Combined"],
})

results_full.round(4)

,Approach,RMSE,MAE,Precision@K,Recall@K,NDCG,Coverage,Diversity,Serendipity,Context
0,Non-Personalized,1.0268,0.8010,0.7000,0.2420,0.8894,0.0253,0.6598,0.0992,No
1,Collaborative Filtering,1.0714,0.8444,0.6069,0.2260,0.8856,0.0696,0.6629,0.1083,No
2,Content-Based,1.1072,0.8661,0.5957,0.2217,0.8654,0.0727,0.6044,0.0660,No
3,Context-Aware,1.1066,0.8659,0.5957,0.2244,0.8675,0.0734,0.6149,0.0673,Weekday/Weekend + Time
4,Hybrid,1.0716,0.8418,0.6966,0.2449,0.8868,0.0290,0.6087,0.0713,Combined


## Final Summary

This notebook presents the evaluation phase of the recommender systems project.

We evaluated the four recommenders using multiple metric families: prediction accuracy (RMSE, MAE), ranking quality (Precision@K, Recall@K, NDCG), and beyond-accuracy metrics (Coverage, Diversity, Serendipity).

A fifth model — the Hybrid recommender — was added by combining the normalised scores
of all four approaches using configurable weights.  A cold-start variant adjusts these
weights dynamically based on how many ratings a user has in the training set, ensuring
that new users still receive reasonable recommendations.

An explainability layer was also implemented: for any user-movie pair the system can
generate plain-text reasons drawn from content similarity, collaborative signals, and
temporal context.

The evaluation was conducted offline using a temporal train/test split. Using a temporal
split is more realistic than a random split because models must always predict future
interactions from past data.

We also included a simulated A/B test design to compare two candidate recommenders
before deployment.

Overall, this notebook provides a unified framework to compare all five recommendation
approaches and serves as the basis for the final interpretation in the report.